In [ ]:
# MarketMind Intelligence Platform V1
# Author: Sharique Mohammad
# Date: April 23, 2026

# MarketMind V1 - Macro Indicators Analysis

**Notebook:** 03_macro_indicators_analysis.ipynb  
**Purpose:** Analyze US economic indicators with time series visualizations

---

## Overview

This notebook analyzes macro economic data from `gold.macro_indicators`:
- CPI and inflation trends
- Unemployment rate analysis
- Federal funds rate (interest rates)
- ADP employment changes
- Multi-indicator comparisons

**Data Period:** 1970s - 2025 (historical dataset)
**Indicators:** 5 key US economic metrics
**Records:** 2,594 data points

In [ ]:
# Import libraries
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')

## 1. Database Connection & Data Loading

In [ ]:
# PostgreSQL connection
DB_CONFIG = {
    'host': '172.31.32.1',
    'port': 5432,
    'database': 'marketmind_v1',
    'user': 'postgres',
    'password': '0940'
}

# Load macro indicators data
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    indicator_name,
    date,
    value,
    indicator_category,
    value_change,
    value_pct_change,
    year,
    month
FROM gold.macro_indicators
ORDER BY date, indicator_name
"""

df = pd.read_sql(query, conn)
conn.close()

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

print(f'Loaded {len(df):,} records')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Indicators: {df["indicator_name"].nunique()}')
print('\nIndicator breakdown:')
print(df['indicator_name'].value_counts())
df.head(10)

## 2. Latest Economic Snapshot

In [ ]:
# Get latest value for each indicator
latest = df.sort_values('date').groupby('indicator_name').tail(1)[['indicator_name', 'date', 'value', 'indicator_category']]
latest = latest.sort_values('indicator_name')

print('Latest Economic Indicators:')
print('=' * 80)
for _, row in latest.iterrows():
    val = row['value']
    if pd.notna(val):
        print(f"{row['indicator_name']:<30} {val:>10.2f}  ({row['indicator_category']:<15}) as of {row['date'].strftime('%Y-%m-%d')}")
    else:
        print(f"{row['indicator_name']:<30} {'N/A':>10}  ({row['indicator_category']:<15}) as of {row['date'].strftime('%Y-%m-%d')}")
print('=' * 80)

## 3. Inflation Trends - CPI Analysis

In [ ]:
# Filter CPI data
cpi_data = df[df['indicator_name'].str.contains('CPI')].copy()
cpi_data = cpi_data[cpi_data['value'].notna()]

# Plot CPI trends
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: CPI MOM values
for indicator in cpi_data['indicator_name'].unique():
    data = cpi_data[cpi_data['indicator_name'] == indicator]
    ax1.plot(data['date'], data['value'], marker='o', markersize=3, label=indicator, linewidth=2)

ax1.set_ylabel('Month-over-Month Change (%)', fontsize=12, fontweight='bold')
ax1.set_title('US Inflation Trends - CPI Analysis (1970-2025)', fontsize=14, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='black', linewidth=1, linestyle='--')

# Plot 2: Recent CPI (last 5 years)
recent_cpi = cpi_data[cpi_data['date'] >= '2020-01-01']
for indicator in recent_cpi['indicator_name'].unique():
    data = recent_cpi[recent_cpi['indicator_name'] == indicator]
    ax2.plot(data['date'], data['value'], marker='o', markersize=4, label=indicator, linewidth=2.5)

ax2.set_xlabel('Date', fontsize=12, fontweight='bold')
ax2.set_ylabel('Month-over-Month Change (%)', fontsize=12, fontweight='bold')
ax2.set_title('Recent CPI Trends (2020-2025)', fontsize=12, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='black', linewidth=1, linestyle='--')

plt.tight_layout()
plt.show()

## 4. Labor Market - Unemployment Rate

In [ ]:
# Filter unemployment data
unemployment = df[df['indicator_name'] == 'US_UNEMPLOYMENT_RATE'].copy()
unemployment = unemployment[unemployment['value'].notna()]

# Plot unemployment rate
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(unemployment['date'], unemployment['value'], linewidth=2.5, color='darkred', label='Unemployment Rate')
ax.fill_between(unemployment['date'], unemployment['value'], alpha=0.3, color='red')

# Add recession shading (simplified - 2008 crisis and COVID)
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), alpha=0.2, color='gray', label='2008 Recession')
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-04-01'), alpha=0.2, color='gray', label='COVID Recession')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Unemployment Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('US Unemployment Rate (1970-2025)', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Unemployment Statistics:')
print(f'  Current: {unemployment.iloc[-1]["value"]:.1f}%')
print(f'  Historical Average: {unemployment["value"].mean():.1f}%')
print(f'  Peak: {unemployment["value"].max():.1f}% on {unemployment.loc[unemployment["value"].idxmax(), "date"].strftime("%Y-%m-%d")}')
print(f'  Lowest: {unemployment["value"].min():.1f}% on {unemployment.loc[unemployment["value"].idxmin(), "date"].strftime("%Y-%m-%d")}')

## 5. Interest Rates - Federal Funds Rate

In [ ]:
# Filter federal funds rate data
fed_rate = df[df['indicator_name'] == 'US_FEDERAL_FUNDS_RATE'].copy()
fed_rate = fed_rate[fed_rate['value'].notna()]

# Plot fed rate
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(fed_rate['date'], fed_rate['value'], linewidth=2.5, color='darkblue', label='Federal Funds Rate')
ax.fill_between(fed_rate['date'], fed_rate['value'], alpha=0.3, color='blue')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Federal Funds Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('US Federal Funds Rate (1982-2025)', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Federal Funds Rate Statistics:')
print(f'  Current: {fed_rate.iloc[-1]["value"]:.2f}%')
print(f'  Historical Average: {fed_rate["value"].mean():.2f}%')
print(f'  Peak: {fed_rate["value"].max():.2f}% on {fed_rate.loc[fed_rate["value"].idxmax(), "date"].strftime("%Y-%m-%d")}')
print(f'  Lowest: {fed_rate["value"].min():.2f}% on {fed_rate.loc[fed_rate["value"].idxmin(), "date"].strftime("%Y-%m-%d")}')

## 6. ADP Employment Changes

In [ ]:
# Filter ADP employment data
adp = df[df['indicator_name'] == 'US_ADP_EMPLOYMENT'].copy()
adp = adp[adp['value'].notna()]

# Plot ADP employment changes
fig, ax = plt.subplots(figsize=(14, 7))

colors = ['green' if x > 0 else 'red' for x in adp['value']]
ax.bar(adp['date'], adp['value'], color=colors, alpha=0.6, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=1.5)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Employment Change (thousands)', fontsize=12, fontweight='bold')
ax.set_title('ADP Employment Changes (2001-2025)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('ADP Employment Statistics:')
print(f'  Latest: {adp.iloc[-1]["value"]:.0f}K jobs')
print(f'  Average Monthly Change: {adp["value"].mean():.0f}K jobs')
print(f'  Largest Gain: {adp["value"].max():.0f}K jobs on {adp.loc[adp["value"].idxmax(), "date"].strftime("%Y-%m-%d")}')
print(f'  Largest Loss: {adp["value"].min():.0f}K jobs on {adp.loc[adp["value"].idxmin(), "date"].strftime("%Y-%m-%d")}')

## 7. Multi-Indicator Comparison (2020-2025)

In [ ]:
# Filter recent data (last 5 years)
recent = df[df['date'] >= '2020-01-01'].copy()
recent = recent[recent['value'].notna()]

# Normalize indicators to 0-100 scale for comparison
fig, ax = plt.subplots(figsize=(14, 8))

for indicator in recent['indicator_name'].unique():
    data = recent[recent['indicator_name'] == indicator].copy()
    if len(data) > 0:
        # Normalize to 0-100 scale
        min_val = data['value'].min()
        max_val = data['value'].max()
        if max_val != min_val:
            data['normalized'] = ((data['value'] - min_val) / (max_val - min_val)) * 100
            ax.plot(data['date'], data['normalized'], marker='o', markersize=3, label=indicator, linewidth=2)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Normalized Value (0-100)', fontsize=12, fontweight='bold')
ax.set_title('Multi-Indicator Comparison - Normalized (2020-2025)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Note: All indicators normalized to 0-100 scale for visual comparison')

## 8. Annual Summary Statistics

In [ ]:
# Calculate annual averages for each indicator
df_clean = df[df['value'].notna()].copy()
annual = df_clean.groupby(['year', 'indicator_name'])['value'].mean().reset_index()

# Focus on recent years (2015-2025)
recent_annual = annual[annual['year'] >= 2015]

# Create pivot table
pivot = recent_annual.pivot(index='year', columns='indicator_name', values='value')

print('Annual Average Economic Indicators (2015-2025):')
print('=' * 120)
print(pivot.to_string())
print('=' * 120)

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot.T, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Annual Economic Indicators Heatmap (2015-2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Indicator', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()